**TASK1**: Dataset: Use the provided dataset containing shopping baskets with grocery items: store_data.csv

In [1]:
from google.colab import drive

# Mount Google Drive - please authorize when prompted
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/store_data.csv', header=None)

print("Dataset 'store_data.csv' loaded successfully into DataFrame 'df'.")
print(f"Original dataset head:\n{df.head()}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/store_data.csv'

**TASK2**:
Random Sampling: Select a 95% random sample of the dataset using the numeric part of your university UID as random_state. Use only this sample for all tasks that follow.
🔴 Penalty: -5 points if you fail to create the random sample correctly or fail to use it for all analysis.

In [ ]:
# Use the provided random_state for sampling
numeric_uid = 70864697

# Select a 95% random sample of the dataset
sampled_df = df.sample(frac=0.95, random_state=numeric_uid)

print(f"\nCreated a 95% random sample of the dataset with random_state = {numeric_uid}.")
print(f"Original dataset size: {len(df)} rows")
print(f"Sampled dataset size: {len(sampled_df)} rows")
print("This 'sampled_df' will be used for all subsequent tasks.")


Created a 95% random sample of the dataset with random_state = 70864697.
Original dataset size: 7501 rows
Sampled dataset size: 7126 rows
This 'sampled_df' will be used for all subsequent tasks.


**TASK 3**:
(2 pts) Generate at least 5 rules with a min_confidence of your choosing. Tweak the value until you get a manageable number of meaningful rules - this applies to all subsequent tasks.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd

# --- Data Preprocessing for Apriori ---
# Ensure the data is in the correct format (list of lists of items)

# Check if the first row of `sampled_df` contains a header-like entry like 'Itemlist'.
# If so, skip it for transaction processing.
if not sampled_df.empty and sampled_df.iloc[0, 0].lower() == 'itemlist':
    transactions_data = sampled_df.iloc[1:, 0] # Skip the first row as it's a header
else:
    transactions_data = sampled_df.iloc[:, 0] # Use all rows if no header-like entry is found

# Convert each string of items into a list of individual items
# Also, clean up any empty strings that might result from splitting
transactions = transactions_data.apply(lambda x: [item.strip() for item in x.strip('"').split(',') if item.strip() != '']).tolist()

# One-hot encode the transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print("Data preprocessed and one-hot encoded successfully. First 5 rows:")
display(df_encoded.head())

# --- TASK 3: Generate at least 5 rules with a min_confidence of your choosing ---

# Find frequent itemsets using the Apriori algorithm
# Start with a min_support. You might need to adjust this value.
# A lower support value will yield more frequent itemsets.
min_support_value = 0.01 # Adjusted to find more rules
frequent_itemsets = apriori(df_encoded, min_support=min_support_value, use_colnames=True)

# Generate association rules from the frequent itemsets
# Start with a min_confidence. You might need to adjust this value.
# A lower confidence value will yield more rules.
min_confidence_value = 0.3 # Adjusted to find more rules
rules_task3 = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence_value)

print(f"\nGenerated {len(rules_task3)} association rules for Task 3 with min_support={min_support_value} and min_confidence={min_confidence_value}.")

if not rules_task3.empty:
    print("\nTop 5 rules by Lift (ordered for insight):")
    # Display rules sorted by lift for better insight into stronger relationships
    display(rules_task3.sort_values(by='lift', ascending=False).head())
else:
    print("No rules found with the current min_support and min_confidence. Please consider lowering them.")

print("\n--- Guidance for Task 3 ---")
print("To generate at least 5 meaningful rules, you may need to **tweak** the `min_support_value` and `min_confidence_value` in the code above. For example, if you get too few rules, try lowering `min_support_value` (e.g., to 0.005) or `min_confidence_value` (e.g., to 0.15), and then re-run this cell. Continue adjusting until you find at least 5 rules that you consider meaningful.")

Data preprocessed and one-hot encoded successfully. First 5 rows:


,Itemlist,almonds,antioxidant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat,whole wheat pasta,yams,yogurt cake,zucchini
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False



Generated 15 association rules for Task 3 with min_support=0.01 and min_confidence=0.3.

Top 5 rules by Lift (ordered for insight):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4,(herb & pepper),(ground beef),0.049537,0.099495,0.016138,0.325779,3.274332,1.0,0.011209,1.335623,0.730796,0.121436,0.251286,0.243990
8,(whole wheat pasta),(milk),0.030031,0.128263,0.010104,0.336449,2.623121,1.0,0.006252,1.313745,0.637932,0.068182,0.238817,0.207612
7,(soup),(milk),0.050519,0.128263,0.015436,0.305556,2.382264,1.0,0.008957,1.255302,0.611104,0.094502,0.203379,0.212953
5,(ground beef),(spaghetti),0.099495,0.173590,0.039714,0.399154,2.299409,1.0,0.022442,1.375411,0.627543,0.170174,0.272944,0.313967
10,(red wine),(spaghetti),0.028347,0.173590,0.010385,0.366337,2.110360,1.0,0.005464,1.304179,0.541497,0.054212,0.233234,0.213079



--- Guidance for Task 3 ---
To generate at least 5 meaningful rules, you may need to **tweak** the `min_support_value` and `min_confidence_value` in the code above. For example, if you get too few rules, try lowering `min_support_value` (e.g., to 0.005) or `min_confidence_value` (e.g., to 0.15), and then re-run this cell. Continue adjusting until you find at least 5 rules that you consider meaningful.


**RESULT:** Generated 15 association rules for Task 3 with min_support=0.01 and min_confidence=0.3.

**TASK 4:** (2 pt) Generate at least 2 rules with one antecedent of your choice.

In [ ]:
from mlxtend.frequent_patterns import association_rules

# --- TASK 4: Generate at least 2 rules with one antecedent of your choice ('herb & pepper') ---

# Define min_support and min_confidence for this task, as requested by the user
# These values can be tweaked to generate a manageable number of meaningful rules
min_support_task4 = 0.01 # Adjusted to find more rules
min_confidence_task4 = 0.1 # Adjusted to find more rules

# Generate association rules from the frequent itemsets
# Using a lower min_threshold for rules generation initially, then filtering by antecedent
rules_task4_all = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence_task4)

# Filter rules to keep only those where 'herb & pepper' is the antecedent
# Also filter by min_support if it's different from the frequent_itemsets generation
rules_with_herb_pepper_antecedent = rules_task4_all[
    rules_task4_all['antecedents'].apply(lambda x: 'herb & pepper' in x)
].copy()

# Further filter by the min_support_task4 if the generated rules_task4_all did not implicitly handle it
rules_with_herb_pepper_antecedent = rules_with_herb_pepper_antecedent[
    rules_with_herb_pepper_antecedent['support'] >= min_support_task4
]

print(f"Generated {len(rules_with_herb_pepper_antecedent)} association rules for Task 4 with 'herb & pepper' as antecedent, min_support={min_support_task4} and min_confidence={min_confidence_task4}.")

if not rules_with_herb_pepper_antecedent.empty:
    print("\nRules with 'herb & pepper' as antecedent (sorted by lift):")
    display(rules_with_herb_pepper_antecedent.sort_values(by='lift', ascending=False))
else:
    print("No rules found with 'herb & pepper' as antecedent using the current min_support and min_confidence.")
    print("Please consider lowering `min_support_task4` or `min_confidence_task4` and re-running this cell.")

print("\n--- Guidance for Task 4 ---")
print("To generate at least 2 meaningful rules, you may need to **tweak** the `min_support_task4` and `min_confidence_task4` in the code above. For example, if you get too few rules, try lowering `min_support_task4` (e.g., to 0.01) or `min_confidence_task4` (e.g., to 0.1), and then re-run this cell.")


Generated 3 association rules for Task 4 with 'herb & pepper' as antecedent, min_support=0.01 and min_confidence=0.1.

Rules with 'herb & pepper' as antecedent (sorted by lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
157,(herb & pepper),(ground beef),0.049537,0.099495,0.016138,0.325779,3.274332,1.0,0.011209,1.335623,0.730796,0.121436,0.251286,0.243990
171,(herb & pepper),(spaghetti),0.049537,0.173590,0.016138,0.325779,1.876719,1.0,0.007539,1.225726,0.491503,0.077966,0.184157,0.209373
87,(herb & pepper),(eggs),0.049537,0.178782,0.012630,0.254958,1.426081,1.0,0.003774,1.102243,0.314349,0.058556,0.092759,0.162801



--- Guidance for Task 4 ---
To generate at least 2 meaningful rules, you may need to **tweak** the `min_support_task4` and `min_confidence_task4` in the code above. For example, if you get too few rules, try lowering `min_support_task4` (e.g., to 0.01) or `min_confidence_task4` (e.g., to 0.1), and then re-run this cell.


**RESULT**: Generated 3 association rules for Task 4 with 'herb & pepper' as antecedent, min_support=0.01 and min_confidence=0.1.


**TASK 5**: (2 pt) Generate at least 2 rules with one consequent of your choice.

In [ ]:
from mlxtend.frequent_patterns import association_rules

# --- TASK 5: Generate at least 2 rules with one consequent of your choice ('ground beef') ---

# Define min_support and min_confidence for this task
# These values can be tweaked to generate a manageable number of meaningful rules
min_support_task5 = 0.015 # Lowered slightly as ground beef pairs can be more specific
min_confidence_task5 = 0.20

# Generate association rules from the frequent itemsets
rules_task5_all = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence_task5)

# Filter rules to keep only those where 'ground beef' is the consequent
rules_with_ground_beef_consequent = rules_task5_all[
    rules_task5_all['consequents'].apply(lambda x: 'ground beef' in x)
].copy()

# Further filter by min_support
rules_with_ground_beef_consequent = rules_with_ground_beef_consequent[
    rules_with_ground_beef_consequent['support'] >= min_support_task5
]

print(f"Generated {len(rules_with_ground_beef_consequent)} association rules for Task 5 with 'ground beef' as consequent, min_support={min_support_task5} and min_confidence={min_confidence_task5}.")

if not rules_with_ground_beef_consequent.empty:
    print("\nRules with 'ground beef' as consequent (sorted by lift):")
    display(rules_with_ground_beef_consequent.sort_values(by='lift', ascending=False))
else:
    print("No rules found with 'ground beef' as consequent using the current min_support and min_confidence.")
    print("Please consider lowering `min_support_task5` or `min_confidence_task5` and re-running this cell.")

print("\n--- Guidance for Task 5 ---")
print("To generate at least 2 meaningful rules, you may need to **tweak** the `min_support_task5` and `min_confidence_task5` in the code above.")

Generated 2 association rules for Task 5 with 'ground beef' as consequent, min_support=0.015 and min_confidence=0.2.

Rules with 'ground beef' as consequent (sorted by lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
56,(herb & pepper),(ground beef),0.049537,0.099495,0.016138,0.325779,3.274332,1.0,0.011209,1.335623,0.730796,0.121436,0.251286,0.243990
60,(spaghetti),(ground beef),0.173590,0.099495,0.039714,0.228779,2.299409,1.0,0.022442,1.167636,0.683808,0.170174,0.143569,0.313967



--- Guidance for Task 5 ---
To generate at least 2 meaningful rules, you may need to **tweak** the `min_support_task5` and `min_confidence_task5` in the code above.


**RESULT**: Generated 2 association rules for Task 5 with 'ground beef' as consequent, min_support=0.015 and min_confidence=0.2.


**TASK 6**:(3 pt) Generate at least 2 rules with two antecedents of your choice.

In [ ]:
from mlxtend.frequent_patterns import association_rules

# --- TASK 6: Generate at least 2 rules with two antecedents of your choice ---

# Define min_support and min_confidence for this task
# These values can be tweaked to generate a manageable number of meaningful rules
min_support_task6 = 0.005 # Keeping a low min_support
min_confidence_task6 = 0.05 # Keeping a low min_confidence

# Generate association rules from the frequent itemsets
rules_task6_all = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence_task6)

# Filter rules to keep only those where 'chocolate' AND 'spaghetti' are antecedents
rules_with_two_antecedents = rules_task6_all[
    rules_task6_all['antecedents'].apply(lambda x: 'chocolate' in x and 'spaghetti' in x)
].copy()

# Further filter by min_support
rules_with_two_antecedents = rules_with_two_antecedents[
    rules_with_two_antecedents['support'] >= min_support_task6
]

print(f"Generated {len(rules_with_two_antecedents)} association rules for Task 6 with 'chocolate' and 'spaghetti' as antecedents, min_support={min_support_task6} and min_confidence={min_confidence_task6}.")

if not rules_with_two_antecedents.empty:
    # Display at least 2 rules, sorted by lift for insight
    print("\nRules with 'chocolate' and 'spaghetti' as antecedents (sorted by lift):")
    display(rules_with_two_antecedents.sort_values(by='lift', ascending=False).head(max(2, len(rules_with_two_antecedents))))
else:
    print("No rules found with 'chocolate' and 'spaghetti' as antecedents using the current min_support and min_confidence.")
    print("Please consider lowering `min_support_task6` or `min_confidence_task6` and re-running this cell.")

print("\n--- Guidance for Task 6 ---")
print("To generate at least 2 meaningful rules, you may need to **tweak** the `min_support_task6` and `min_confidence_task6` in the code above. For example, if you get too few rules, try lowering `min_support_task6` (e.g., to 0.001) or `min_confidence_task6` (e.g., to 0.01), and then re-run this cell.")


Generated 2 association rules for Task 6 with 'chocolate' and 'spaghetti' as antecedents, min_support=0.005 and min_confidence=0.05.

Rules with 'chocolate' and 'spaghetti' as antecedents (sorted by lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
274,"(chocolate, spaghetti)",(milk),0.038731,0.128263,0.010104,0.260870,2.033869,1.0,0.005136,1.179409,0.528808,0.064401,0.152118,0.169822
268,"(chocolate, spaghetti)",(eggs),0.038731,0.178782,0.010665,0.275362,1.540213,1.0,0.003741,1.133281,0.364871,0.051560,0.117606,0.167508



--- Guidance for Task 6 ---
To generate at least 2 meaningful rules, you may need to **tweak** the `min_support_task6` and `min_confidence_task6` in the code above. For example, if you get too few rules, try lowering `min_support_task6` (e.g., to 0.001) or `min_confidence_task6` (e.g., to 0.01), and then re-run this cell.


**RESULT**: Generated 2 association rules for Task 6 with 'chocolate' and 'spaghetti' as antecedents, min_support=0.005 and min_confidence=0.05

**TASK 7:** (3 pt) Generate at least 2 rules with two consequents of your choice.

**Reasoning**:
I need to implement the logic for Task 7, which involves generating association rules, filtering them for exactly two consequents, and potentially adjusting `min_support` and `min_confidence` if fewer than two rules are found. This will be done in a single code block.



In [ ]:
from mlxtend.frequent_patterns import association_rules, apriori
import pandas as pd # Ensure pandas is imported for DataFrame operations

# --- TASK 7: Generate at least 2 rules with two consequents of your choice ---

# Re-initialize frequent_itemsets here to ensure it's defined even if previous cells weren't run.
# Using the min_support_value from Task 3 (0.01) as a starting point for robustness.
# This assumes df_encoded is available from a previous cell. If not, earlier data preprocessing cells must be run.
min_support_for_initial_frequent_itemsets = 0.01
frequent_itemsets = apriori(df_encoded, min_support=min_support_for_initial_frequent_itemsets, use_colnames=True)

# 1. Initialize min_support_task7 and min_confidence_task7
min_support_task7 = 0.01 # Initial min_support (1%)
min_confidence_task7 = 0.05 # Initial min_confidence (5%)

# 2. Define a helper function, generate_and_filter_two_consequent_rules
def generate_and_filter_two_consequent_rules(freq_itemsets, min_conf, min_supp_for_rules):
    # a. Generate association rules from the provided freq_itemsets
    rules = association_rules(freq_itemsets, metric="confidence", min_threshold=min_conf)

    # b. Filter these rules to keep only those where the 'consequents' column contains exactly two items.
    rules_two_consequents = rules[rules['consequents'].apply(lambda x: len(x) == 2)].copy()

    # c. Further filter these rules to ensure their 'support' is greater than or equal to min_supp_for_rules.
    rules_two_consequents = rules_two_consequents[
        rules_two_consequents['support'] >= min_supp_for_rules
    ]
    return rules_two_consequents

# 3. Call the generate_and_filter_two_consequent_rules function initially
rules_with_two_consequents = generate_and_filter_two_consequent_rules(frequent_itemsets, min_confidence_task7, min_support_task7)

print(f"Initial attempt for Task 7 (min_support={min_support_task7}, min_confidence={min_confidence_task7}): Found {len(rules_with_two_consequents)} rules.")

# 4. Check if the number of rules found is less than 2. If it is:
if len(rules_with_two_consequents) < 2:
    print("\nLess than 2 rules found. Regenerating frequent itemsets and re-evaluating rules...")
    # a. Set min_support_for_apriori to 0.005.
    min_support_for_apriori = 0.005 # Lower support for Apriori

    # b. Regenerate frequent_itemsets
    frequent_itemsets_re_generated = apriori(df_encoded, min_support=min_support_for_apriori, use_colnames=True)

    # c. Set min_confidence_task7 to 0.01
    min_confidence_task7 = 0.01 # Lower confidence for rules

    # d. Call generate_and_filter_two_consequent_rules again
    rules_with_two_consequents = generate_and_filter_two_consequent_rules(
        frequent_itemsets_re_generated, min_confidence_task7, min_support_for_apriori
    )

    print(f"Second attempt for Task 7 (min_support={min_support_for_apriori}, min_confidence={min_confidence_task7}): Found {len(rules_with_two_consequents)} rules.")

    # e. If the number of rules is still less than 2, lower min_confidence_task7 further to 0.005
    if len(rules_with_two_consequents) < 2:
        print("Still less than 2 rules found. Lowering min_confidence further...")
        min_confidence_task7 = 0.005 # Even lower confidence
        rules_with_two_consequents = generate_and_filter_two_consequent_rules(
            frequent_itemsets_re_generated, min_confidence_task7, min_support_for_apriori
        )
        print(f"Third attempt for Task 7 (min_support={min_support_for_apriori}, min_confidence={min_confidence_task7}): Found {len(rules_with_two_consequents)} rules.")

    # f. Update min_support_task7 and frequent_itemsets if they were regenerated
    min_support_task7 = min_support_for_apriori
    frequent_itemsets = frequent_itemsets_re_generated # Update the global frequent_itemsets

# 5. Print the total number of rules generated for Task 7
print(f"\nGenerated {len(rules_with_two_consequents)} association rules for Task 7 with exactly two consequents (min_support={min_support_task7} and min_confidence={min_confidence_task7}).")

# 6. If rules_with_two_consequents is not empty, display the rules
if not rules_with_two_consequents.empty:
    print("\nRules with exactly two consequents (sorted by lift):")
    display(rules_with_two_consequents.sort_values(by='lift', ascending=False).head(max(2, len(rules_with_two_consequents))))
else:
    print("No rules found with exactly two consequents using the current min_support and min_confidence.")
    print("Please consider lowering the min_support_task7 or min_confidence_task7 values further.")

Initial attempt for Task 7 (min_support=0.01, min_confidence=0.05): Found 6 rules.

Generated 6 association rules for Task 7 with exactly two consequents (min_support=0.01 and min_confidence=0.05).

Rules with exactly two consequents (sorted by lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
275,(milk),"(chocolate, spaghetti)",0.128263,0.038731,0.010104,0.078775,2.033869,1.0,0.005136,1.043467,0.583119,0.064401,0.041657,0.169822
277,(spaghetti),"(milk, chocolate)",0.173590,0.031013,0.010104,0.058205,1.876793,1.0,0.004720,1.028873,0.565308,0.051948,0.028062,0.191999
271,(spaghetti),"(eggs, chocolate)",0.173590,0.033820,0.010665,0.061439,1.816656,1.0,0.004794,1.029427,0.543965,0.054208,0.028586,0.188396
276,(chocolate),"(milk, spaghetti)",0.163766,0.034521,0.010104,0.061697,1.787197,1.0,0.004450,1.028962,0.526724,0.053691,0.028147,0.177190
270,(chocolate),"(eggs, spaghetti)",0.163766,0.036767,0.010665,0.065124,1.771280,1.0,0.004644,1.030333,0.520712,0.056171,0.029440,0.177600
269,(eggs),"(chocolate, spaghetti)",0.178782,0.038731,0.010665,0.059655,1.540213,1.0,0.003741,1.022251,0.427096,0.051560,0.021766,0.167508


In [ ]:
from mlxtend.frequent_patterns import association_rules, apriori
import pandas as pd

# --- New Task: Generate rules with specific two consequents ('chocolate' and 'spaghetti') ---

# 1. Define a helper function, generate_and_filter_rules_two_consequents
def generate_and_filter_rules_two_consequents(freq_itemsets, min_conf, min_supp_for_rules):
    # a. Generate association rules from the provided freq_itemsets
    rules = association_rules(freq_itemsets, metric="confidence", min_threshold=min_conf)

    # b. Filter these rules to include only those where both 'chocolate' AND 'spaghetti' are present in the 'consequents' column.
    rules_specific_consequents = rules[
        rules['consequents'].apply(lambda x: 'chocolate' in x and 'spaghetti' in x)
    ].copy()

    # c. Further filter these rules to ensure their 'support' is greater than or equal to the provided minimum support.
    rules_specific_consequents = rules_specific_consequents[
        rules_specific_consequents['support'] >= min_supp_for_rules
    ]
    return rules_specific_consequents

# 2. Initialize min_support_task_new and min_confidence_task_new
min_support_task_new = 0.01 # Initial min_support (1%)
min_confidence_task_new = 0.05 # Initial min_confidence (5%)

# 3. Call the generate_and_filter_rules_two_consequents function initially
rules_with_choc_spag_consequents = generate_and_filter_rules_two_consequents(frequent_itemsets, min_confidence_task_new, min_support_task_new)

print(f"Initial attempt: Found {len(rules_with_choc_spag_consequents)} rules with 'chocolate' and 'spaghetti' as consequents (min_support={min_support_task_new}, min_confidence={min_confidence_task_new}).")

# 4. Check if the number of rules found is less than 2. If it is:
if len(rules_with_choc_spag_consequents) < 2:
    print("\nLess than 2 rules found. Regenerating frequent itemsets and re-evaluating rules...")
    # a. Set min_support_apriori_low to 0.005.
    min_support_apriori_low = 0.005 # Lower support for Apriori

    # b. Regenerate frequent_itemsets
    # Assuming df_encoded is available from previous cells
    frequent_itemsets_re_generated = apriori(df_encoded, min_support=min_support_apriori_low, use_colnames=True)

    # c. Set min_confidence_task_new to 0.01.
    min_confidence_task_new = 0.01 # Lower confidence for rules

    # d. Call generate_and_filter_rules_two_consequents again
    rules_with_choc_spag_consequents = generate_and_filter_rules_two_consequents(
        frequent_itemsets_re_generated, min_confidence_task_new, min_support_apriori_low
    )

    print(f"Second attempt: Found {len(rules_with_choc_spag_consequents)} rules (min_support={min_support_apriori_low}, min_confidence={min_confidence_task_new}).")

    # e. If the number of rules is still less than 2, lower min_confidence_task_new further to 0.005
    if len(rules_with_choc_spag_consequents) < 2:
        print("Still less than 2 rules found. Lowering min_confidence further...")
        min_confidence_task_new = 0.005 # Even lower confidence
        rules_with_choc_spag_consequents = generate_and_filter_rules_two_consequents(
            frequent_itemsets_re_generated, min_confidence_task_new, min_support_apriori_low
        )
        print(f"Third attempt: Found {len(rules_with_choc_spag_consequents)} rules (min_support={min_support_apriori_low}, min_confidence={min_confidence_task_new}).")

    # Update frequent_itemsets global variable if it was regenerated, for consistency with later tasks
    frequent_itemsets = frequent_itemsets_re_generated

# 5. Print the final number of rules generated and display the DataFrame
print(f"\nFinal count: Generated {len(rules_with_choc_spag_consequents)} association rules with 'chocolate' and 'spaghetti' as consequents (min_support used: {min_support_task_new if len(rules_with_choc_spag_consequents) < 2 else 0.01} and min_confidence used: {min_confidence_task_new}).")

if not rules_with_choc_spag_consequents.empty:
    print("\nRules with 'chocolate' and 'spaghetti' as consequents (sorted by lift):")
    pd.set_option('display.max_columns', None) # Display all columns
    display(rules_with_choc_spag_consequents.sort_values(by='lift', ascending=False))
    pd.reset_option('display.max_columns') # Reset option
else:
    print("No rules found with 'chocolate' and 'spaghetti' as consequents using the adjusted parameters.")


Initial attempt: Found 2 rules with 'chocolate' and 'spaghetti' as consequents (min_support=0.01, min_confidence=0.05).

Final count: Generated 2 association rules with 'chocolate' and 'spaghetti' as consequents (min_support used: 0.01 and min_confidence used: 0.05).

Rules with 'chocolate' and 'spaghetti' as consequents (sorted by lift):


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
275,(milk),"(chocolate, spaghetti)",0.128263,0.038731,0.010104,0.078775,2.033869,1.0,0.005136,1.043467,0.583119,0.064401,0.041657,0.169822
269,(eggs),"(chocolate, spaghetti)",0.178782,0.038731,0.010665,0.059655,1.540213,1.0,0.003741,1.022251,0.427096,0.051560,0.021766,0.167508


**RESULT**: Generated 2 association rules with 'chocolate' and 'spaghetti' as consequents (min_support used: 0.01 and min_confidence used: 0.05).


**TASK 8**: (3 pt) Generate at least 1 rule with two antecedents and one consequent of your choice.

In [ ]:
# 1. Set a reachable support for the 3-item combo (chocolate, spaghetti, ground beef)
min_support_task8 = 0.001
min_conf_task8 = 0.05

# 2. Run Apriori once (limiting max_len to 3 for speed and memory management)
frequent_itemsets_task8 = apriori(df_encoded,
                                  min_support=min_support_task8,
                                  use_colnames=True,
                                  max_len=3)

# 3. Generate all rules
all_rules_task8 = association_rules(frequent_itemsets_task8,
                                    metric="confidence",
                                    min_threshold=min_conf_task8)

# 4. Target the specific rule: {chocolate, spaghetti} -> {ground beef}
task8_final = all_rules_task8[
    (all_rules_task8['antecedents'] == frozenset({'chocolate', 'spaghetti'})) &
    (all_rules_task8['consequents'] == frozenset({'ground beef'}))
]

if not task8_final.empty:
    print(f"Success! Found {len(task8_final)} rule(s).")
    display(task8_final)
else:
    print("No exact match found for {chocolate, spaghetti} -> {ground beef}.")
    print("Try lowering min_support_task8 to 0.0005 or checking if the combination exists in your sample.")

Success! Found 1 rule(s).


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
5569,"(chocolate, spaghetti)",(ground beef),0.038731,0.099495,0.009262,0.23913,2.403446,1.0,0.005408,1.183521,0.607459,0.071817,0.155064,0.16611


**RESULT**: Generated 1 association rule with 'chocolate' and 'ground beef' as antecedents and eggs as a consequent (min_support used: 0.001 and min_confidence used: 0.05).

**TASK 9:** 5 pts) At the end of the notebook, add a plain text cell answering:
What interesting insights did you find from your analysis?

## Deeper Insights from Association Rules Analysis

Beyond just identifying which items are frequently bought together, a closer look at the metrics reveals interesting properties and more specific behavioral patterns:

1.  **Asymmetry in Confidence for Reciprocal Rules (e.g., `ground beef <-> spaghetti`):**
    *   **Rule 1:** `(ground beef) -> (spaghetti)`
    *   **Rule 2:** `(spaghetti) -> (ground beef)`
    *   **Insight:** While the **support** (how often they appear together) and **lift** (how much more likely they are to appear together than by chance) are identical for both directions of the rule, the **confidence** levels are significantly different. This is because confidence is not symmetric and depends on the frequency of the antecedent. A customer buying ground beef is more likely to also pick up spaghetti than a customer buying spaghetti is to pick up ground beef, despite these two items being strongly associated.

2.  **Implications of Asymmetry with Multiple Items (e.g., `{chocolate, spaghetti} <-> {milk}`):**
    *   **Rule 1:** `({chocolate, spaghetti}) -> ({milk})`
    *   **Rule 2:** `({milk}) -> ({chocolate, spaghetti})`
    *   **Insight:** Similarly, the confidence that `milk` will be bought given `chocolate` and `spaghetti` is considerably higher than the confidence that `chocolate` and `spaghetti` will be bought given `milk`. This suggests that customers who have already decided on a more specific combination like 'chocolate and spaghetti' are quite likely to add 'milk', possibly for a related recipe. Conversely, a customer buying 'milk' has a broader range of potential complementary purchases, making the specific `chocolate` and `spaghetti` combination less certain.

**Overall Takeaway:**

The difference in confidence values for reciprocal rules, while lift and support remain constant, provides crucial directional insights. It helps us understand which item is a stronger predictor of the other, allowing for more targeted marketing strategies (e.g., promoting spaghetti when ground beef is in the cart, rather than the other way around if shelf space is limited for cross-promotions).

## Further Insights into Association Rules

1.  **The Nuance of High Lift with Low Support:**
    *   **Insight:** A rule can have a very high `lift` value, indicating a strong positive correlation between the items, but still have relatively `low support`. For example, a rule like `(burgers, eggs) -> (cooking oil)` might show a high lift, suggesting that if customers buy burgers and eggs, they are significantly more likely to buy cooking oil than expected by chance. However, if the `support` for this rule is very low (e.g., 0.001), it means this specific combination of purchases happens very infrequently. While statistically interesting, such a rule might not be as actionable from a business perspective as a rule with moderately high lift but much higher support, which represents a larger segment of transactions.
    *   **Actionable Takeaway:** Businesses often need to balance statistical significance (high lift) with practical importance (sufficient support) when deciding on promotional strategies or store layouts. A rule with low support, even if it has high lift, might not justify a major change if it only applies to a tiny fraction of customers.

2.  **Popular Items vs. True Associations (High Support, Moderate Lift):**
    *   **Insight:** Some items are simply very popular and are bought frequently regardless of other items. For instance, 'mineral water' or 'milk' might appear in many rules as either an antecedent or a consequent. When a rule involves such a popular item, its `confidence` can be relatively high (e.g., `(avocado) -> (mineral water)`), but its `lift` might be closer to 1 (e.g., 1.1 or 1.2). A lift close to 1 suggests that the items are not purchased together much more frequently than one would expect given their individual popularity.
    *   **Actionable Takeaway:** Rules with moderate lift involving highly popular items might indicate general co-occurrence rather than a strong, specific purchasing impulse. For instance, a customer buying 'avocado' might coincidentally also buy 'mineral water' because they buy mineral water all the time anyway. These rules are less indicative of strong cross-selling opportunities and more reflective of overall customer buying habits for staples.